# DWT Application Example: Audio file

Download a .wav file.

Decompose the signal in low and high frequencies by using DWT.

Then write the following files:

1. The fully reconstructed signal from the approximation and detail coefficients.


2. The low-frequency signal considering the approximation coefficients only.


3. The high-frequency signal considering the detail coefficients only.


4. Listen to all three audio tracks and compare them to the original one. What do you find?

### Reference file:
https://github.com/helgadenes/Computational_Physics_USFQ/blob/main/Unit_5/ImperialMarch60.wav

In [ ]:
from scipy.io import wavfile
import IPython.display as ipd
import numpy as np
import matplotlib.pyplot as plt
import pywt

### Read wav file:

In [ ]:
# Define file path

filepath = "./"

# File I/O 
samplerate, data = wavfile.read(filepath + "ImperialMarch60.wav")

In [ ]:
# Check the signal

print(type(samplerate))

print(data.shape)

print(data, np.min(data), np.max(data))

In [ ]:
#ipd.Audio(filepath + "ImperialMarch60.wav")

### Define x and y axes:

In [ ]:
# Generate the time axis using the sampling rate
t = np.arange(len(data))/float(samplerate)

# Normalising the signal
data = data/np.max(data)

### Plot signal:

In [ ]:
# Plotting the audio track

plt.figure(figsize=(11,3))

plt.plot(t, data)

plt.xlabel(r"$t$")
plt.ylabel(r"$I$")
#plt.xlim(0,2)
plt.show()

In [ ]:
# Plotting the audio track

plt.figure(figsize=(11,4))

plt.plot(t, data)
plt.xlim(15.5,15.6)

plt.xlabel(r"$t$")
plt.ylabel(r"$I$")
plt.show()

### DWT to get coefficients:

In [ ]:
cA, cD = pywt.dwt(data, 'bior6.8', 'per')

print(cA.shape, cD.shape, data.shape)

In [ ]:
# Applying the iDWT
data_reconstruted = pywt.idwt(cA, cD, 'bior6.8', 'per')

# Saving the reconstructed file
wavfile.write(filepath + 'full_reconstructed.wav', samplerate, data_reconstruted)


In [ ]:
#wavfile.write('../sample-data/reconstructed.wav', samplerate, data_reconstruted)



### Comparing the original and the reconstructured signals:

In [ ]:
plt.figure(figsize=(11,4))

plt.plot(t, data, "blue")
plt.plot(t, data_reconstruted, linestyle = ":", c="yellow")
plt.xlim(15.5,15.6)

plt.xlabel(r"$t$")
plt.ylabel(r"$I$")
plt.show()

In [ ]:
samplerate1, data1 = wavfile.read(filepath + "full_reconstructed.wav")

print(samplerate1)

print(data1.shape)

In [ ]:
#ipd.Audio(filepath + "full_reconstructed.wav")

#### Isolate the low frequencies (Approximation coefficients)



In [ ]:
# Saving the low freq. directly
wavfile.write(filepath + 'only_ca.wav', samplerate, cA) 

# 2nd try:
data_low = pywt.idwt(cA, None, "bior6.8", "per")

# Formatting: https://docs.scipy.org/doc/scipy/reference/generated/scipy.io.wavfile.read.html

data_low_f = np.clip(data_low, -1., +1.).astype(np.float32)
 
wavfile.write(filepath + 'only_ca2.wav', samplerate, data_low)
wavfile.write(filepath + 'only_ca3.wav', samplerate, data_low_f)

In [ ]:
#ipd.Audio(filepath + "only_ca3.wav")

#### Isolate the high frequencies (Approximation coefficients)

In [ ]:
# Saving the low freq. directly
wavfile.write(filepath + 'only_cd.wav', samplerate, cD) 


# 2nd try:
data_high = pywt.idwt(None, cD, "bior6.8", "per")

# Formatting: https://docs.scipy.org/doc/scipy/reference/generated/scipy.io.wavfile.read.html

data_high_f = np.clip(data_high, -1., +1.).astype(np.float32)

wavfile.write(filepath + 'only_cd2.wav', samplerate, data_high)
wavfile.write(filepath + 'only_cd3.wav', samplerate, data_high_f)

In [ ]:
#ipd.Audio(filepath + "only_cd3.wav")

#wavfile.write('../sample-data/with_only_cds.wav', samplerate, cD) #high freq.

### Alternative decomposition with pywt.wavedec():

In [ ]:
coeffs_audio = pywt.wavedec(data, 'bior6.8', level = 2, mode = 'periodic')

#print(len(coeffs_audio))

ca, cd2, cd1 = coeffs_audio

In [ ]:
print("Second level cA: ", ca.shape)
print("First level cD1: ",  cd1.shape)
print("Second level cD2: ", cd2.shape)
#print(cd3.shape)
#print(cd4.shape)

### Recovering the original signal with pywt.waverec()

In [ ]:
# Let's get the original signal again:

recovered_audio = pywt.waverec(coeffs_audio, 'bior6.8', mode = 'periodic')

In [ ]:
wavfile.write(filepath + 'full_reconstructed2.wav', samplerate, recovered_audio)

### Only using cA (2nd level):

In [ ]:
print(len(coeffs_audio))
#print(coeffs_audio[-2])

In [ ]:
# Removing the second level detail coefficients
coeffs_audio[-2] = np.zeros_like(coeffs_audio[-2])

# Removing the first level detail coefficients
coeffs_audio[-1] = np.zeros_like(coeffs_audio[-1])

In [ ]:
# Only cA coeff.

recovered_audio_ca = pywt.waverec(coeffs_audio, 'bior6.8', mode = 'periodic')

In [ ]:
wavfile.write(filepath + 'rec_ca2.wav', samplerate, recovered_audio_ca)

### Alternative decomposition with pywt.wavedec():

In [ ]:
coeffs_audio = pywt.wavedec(data, 'bior6.8', level = 3, mode = 'periodic')

print(len(coeffs_audio))

ca, cd3, cd2, cd1 = coeffs_audio

In [ ]:
print("Third level cA: ", ca.shape)
print("First level cD1: ",  cd1.shape)
print("Second level cD2: ", cd2.shape)
print("Third level cD3: ", cd3.shape)
#print(cd3.shape)
#print(cd4.shape)

In [ ]:
# Removing the third level detail coefficients
coeffs_audio[-3] = np.zeros_like(coeffs_audio[-3])

# Removing the second level detail coefficients
coeffs_audio[-2] = np.zeros_like(coeffs_audio[-2])

# Removing the first level detail coefficients
coeffs_audio[-1] = np.zeros_like(coeffs_audio[-1])

In [ ]:
# Only cA coeff.

recovered_audio_ca = pywt.waverec(coeffs_audio, 'bior6.8', mode = 'periodic')

In [ ]:
wavfile.write(filepath + 'rec_ca3.wav', samplerate, recovered_audio_ca)

### Reconstructing third level high-freq components:

In [ ]:
coeffs_audio = pywt.wavedec(data, 'bior6.8', level = 3, mode = 'periodic')

print(len(coeffs_audio))

ca, cd3, cd2, cd1 = coeffs_audio

In [ ]:
# Removing the third level approx coefficients
coeffs_audio[0] = np.zeros_like(coeffs_audio[0])

# Removing the second level detail coefficients
#coeffs_audio[-2] = np.zeros_like(coeffs_audio[-2])

# Removing the first level detail coefficients
#coeffs_audio[-1] = np.zeros_like(coeffs_audio[-1])

In [ ]:
# Only cA coeff.

recovered_audio_ca = pywt.waverec(coeffs_audio, 'bior6.8', mode = 'periodic')

In [ ]:
wavfile.write(filepath + 'rec_cd3.wav', samplerate, recovered_audio_ca)